## Repository path configuration
All repository files are accessed through relative paths. Run the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import os

# Resolve the repository root whether Jupyter starts in the repository root
# or directly inside the notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_DIR = ROOT / "data"
EXTERNAL_DATA_DIR = DATA_DIR / "external"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT / "outputs"
RESULTS_DIR = OUTPUTS_DIR / "results"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EXTERNAL_MATERIALS_DIR = ROOT / "external_materials"
MODEL_WEIGHTS_DIR = EXTERNAL_MATERIALS_DIR / "model_weights"

for folder in [
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR / "logs",
    RESULTS_DIR,
    FIGURES_DIR,
    MODEL_WEIGHTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)


In [ ]:
import os
os.environ["WANDB_API_KEY"] = os.getenv("WANDB_API_KEY", "")

In [ ]:
!pip install wandb -qU  # install της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login(key=os.environ["WANDB_API_KEY"]) if os.getenv("WANDB_API_KEY") else None

In [ ]:
# Google Drive mounting is intentionally not used in this repository.

In [ ]:
"""
installation of necessary components.
"""
!pip install accelerate -U
!pip install transformers[torch]
!pip install bitsandbytes

In [ ]:
"""
Συμπερίληψη απαραίτων βιβλιοθηκών και Python modules.
"""
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DebertaV2Tokenizer, DebertaForSequenceClassification, DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import RobertaTokenizer, RobertaForSequenceClassification, XLNetTokenizer, XLNetForSequenceClassification
from transformers import ElectraTokenizer, ElectraForSequenceClassification, AlbertTokenizer, AlbertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import time
import random
import bitsandbytes as bnb
from torch.optim import Adam
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
import string
from nltk.corpus import wordnet

# Κατέβασμα των απαραίτητων δεδομένων από το nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')  # Για σωστό lemmatization
nltk.download('averaged_perceptron_tagger')  # POS tagging
nltk.download('punkt')  # Tokenizer
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
# Για αναπαραγωγιμότητα. Αν το seed είναι το ίδιο, παίρνεις την ίδια ακολουθία τυχαίων επιλογών. Θέλω να παίρνω κάθε φορά το ίδιο test set.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
""" Οι συναρτήσεις καθαρισμού σε αυτό το κελί είναι προαιρετικές. Καλύτερα να μην εφαρμόζονται σε LLMs. """

# Αφαίρεση stopwords, σημείων στίξης και lemmatization.
def get_wordnet_pos(word):
    """ Μετατρέπει τις POS tags από το nltk στη μορφή που απαιτεί το WordNet Lemmatizer """
    tag = pos_tag([word])[0][1][0].upper()  # Παίρνουμε το πρώτο γράμμα του POS tag
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}  # Μετατροπή σε WordNet μορφή
    return tag_dict.get(tag, wordnet.NOUN)  # Αν δεν υπάρχει στο dict, υποθέτουμε NOUN (ουσιαστικό)


# Έκδοση της clean_text() με Lemmatization + lowercase
def clean_text(text, language="english"):
    """
    Καθαρίζει ένα κείμενο από stopwords, σημεία στίξης και εκτελεί σωστό lemmatization.
    Οι λέξεις μετατρέπονται σε πεζά γράμματα.
    """
    stop_words = set(stopwords.words(language))
    lemmatizer = WordNetLemmatizer()

    if isinstance(text, str):
        # Μετατροπή όλων των χαρακτήρων σε πεζά
        text = text.lower()

        # Αφαίρεση σημείων στίξης
        text = text.translate(str.maketrans('', '', string.punctuation))

        # Tokenization (Διαχωρισμός σε λέξεις)
        words = word_tokenize(text)

        # Αφαίρεση stopwords
        filtered_words = [word for word in words if word not in stop_words]

        # Lemmatization με βάση το σωστό POS tag
        lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in filtered_words]

        return " ".join(lemmatized_words)

    return text  # Αν δεν είναι string, επιστρέφει το αρχικό δεδομένο.


# Συνάρτηση που εφαρμόζει την clean_text στο dataset.
def remove_stopwords_from_csv(df, text_column="sentence", language="english"):
    """
    Εφαρμόζει την clean_text σε μια στήλη του DataFrame για να αφαιρέσει stopwords, σημεία στίξης και να εκτελέσει lemmatization.

    Args:
        df (pd.DataFrame): Το DataFrame που περιέχει τα δεδομένα.
        text_column (str): Το όνομα της στήλης που περιέχει το κείμενο.
        language (str): Η γλώσσα των stopwords (default: 'english').

    Returns:
        pd.DataFrame: Το καθαρισμένο DataFrame.
    """
    df[text_column] = df[text_column].apply(lambda x: clean_text(x, language))
    return df



""" Στα LLMs καλύτερα εφαρμογή μόνο lowercasing. """
def clean_text_only(text, language="english"):
    text = text.lower()
    return text

def lower_words_only(df, text_column="sentence", language="english"):
    df[text_column] = df[text_column].apply(lambda x: clean_text_only(x, language))
    return df

In [ ]:
"""
Φόρτωση και ανάλυση των δεδομένων.
"""

# Φόρτωση των δεδομένων σε dataframe.
df = pd.read_csv('data/external/sentences_dataset_45269.csv')

In [ ]:
# Εκτύπωση των πρώτων γραμμών πριν την επεξεργασία (καθαρισμός stopwords, lemmatize, καθαρισμός σημείων στίξης).
"""print("\nΠροτάσεις πριν τον καθαρισμό!\n")
print(df.head(20))

# Η lower_words_only εφαρμόζει μόνο lowercasing. Αν θέλω καθαρισμό (stop words, lemmatize) καλώ την remove_stopwords_from_csv.
df = lower_words_only(df, text_column="sentence", language="english")

# Εκτύπωση των πρώτων γραμμών μετά την επεξεργασία (καθαρισμός stopwords, lemmatize, καθαρισμός σημείων στίξης).
print("\nΠροτάσεις μετά τον καθαρισμό!\n")
print(df.head(20))"""

# Εμφάνιση δειγμάτων.
print("\nΠροτάσεις: \n")
print(df.head(30))

In [ ]:
# Αφαίρεση διπλότυπων εγγραφών. Διατηρείται μόνο η πρώτη εμφάνιση της εν λόγω γραμμής.

# Υπολογισμός αριθμού διπλών εγγραφών.
num_duplicates = df['sentence'].duplicated().sum()
print("Πλήθος διπλών εγγραφών στη στήλη sentence:", num_duplicates)

# Αποθήκευση του αρχικού πλήθους γραμμών.
before = len(df)

# Αφαίρεση διπλότυπων (κρατάμε την πρώτη εμφάνιση).
df = df.drop_duplicates(subset='sentence', keep='first')

# Υπολογισμός του πόσες γραμμές διαγράφηκαν.
after = len(df)
deleted = before - after
print("Πλήθος εγγραφών που διαγράφηκαν:", deleted)

In [ ]:
# Υπολογισμός στατιστικών για τη στήλη polarity.
polarity_stats = df['polarity'].value_counts().sort_index()

# Εμφάνιση των αποτελεσμάτων.
print("Polarity Statistics:\n", polarity_stats)

In [ ]:
from matplotlib.patches import Patch

# Ρυθμίσεις εμφάνισης
sns.set(style="whitegrid", context="paper", font_scale=1.0)

# Ορισμός χαρτογράφησης
label_map = {0: "Negative", 1: "Neutral", 2: "Positive"}

plt.rcParams.update({'font.size': 11})

# Δημιουργία ραβδογράμματος
plt.figure(figsize=(6, 4))


ax = sns.barplot(
    x=polarity_stats.index,
    y=polarity_stats.values,
    hue=polarity_stats.index,   # προσθέτουμε hue=x για να φύγει το FutureWarning
    legend=False,               # αλλά απενεργοποιούμε το default legend
    palette=["#89CFF0", "#FA8072", "#90EE90"]
)


# Προσαρμογή ετικετών στον άξονα x
ax.set_xticks(range(len(polarity_stats.index)))
ax.set_xticklabels([label_map[i] for i in polarity_stats.index], fontsize=11)
#ax.set_yticklabels(ax.get_yticks(), fontsize=11)


# Προσθήκη τίτλων και ετικετών
#plt.title('Κατανομή ετικετών polarity', weight='bold')
plt.xlabel('')
plt.ylabel('Number of sentences')

# Προσθήκη αριθμητικών τιμών πάνω στις μπάρες
for p in ax.patches:
    height = p.get_height()
    ax.text(
        p.get_x() + p.get_width() / 2.,  # οριζόντια θέση στο κέντρο της μπάρας
        height + max(polarity_stats.values) * 0.00,  # λίγο πάνω από την μπάρα
        f'{int(height)}',  # τιμή
        ha='center', va='bottom', fontsize=10
    )


# Προσθήκη custom legend (χωρίς warnings)
colors = ["#89CFF0", "#FA8072", "#90EE90"]
labels = [label_map[i] for i in polarity_stats.index]
handles = [Patch(color=c, label=l) for c, l in zip(colors, labels)]
plt.legend(handles=handles, title='Sentiment', loc='upper right', fontsize=11, title_fontsize=11)


# Ρύθμιση πλέγματος και περιθωρίων
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Αποθήκευση ως PDF υψηλής ανάλυσης.
plt.savefig(
    f'outputs/figures/fig/polarity_distribution.pdf',
    format='pdf',
    bbox_inches='tight',
    dpi=300
)

plt.show()

In [ ]:
# Διαίρεση των δεδομένων σε εκπαίδευσης και ελέγχου.
train_data, temp_data = train_test_split(df, test_size=0.3, stratify=df['polarity'], random_state=SEED)
val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=temp_data['polarity'], random_state=SEED)

#temp_data, test_data = train_test_split(df, test_size=0.3, stratify=df['polarity'], random_state=SEED)
#train_data, val_data = train_test_split(temp_data, test_size=0.2, stratify=temp_data['polarity'], random_state=SEED)

# Εμφάνιση των διαστάσεων των σετ δεδομένων.
print("Διαστάσεις Σετ Εκπαίδευσης:", train_data.shape)
print("Διαστάσεις Σετ Ελέγχου:", test_data.shape)

print("Το πρώτο δείγμα του train είναι: ", train_data.head(1))

In [ ]:
# Κλάση για τη διαχείριση των δεδομένων
class PolarityDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


# Επιλογή μεταξύ 7 μοντέλων.
print("Επιλέξτε μοντέλο: 1 - BERT, 2 - DeBERTa, 3 - XLNet, 4 - RoBERTa, 5 - DistilBERT, 6 - ELECTRA, 7 - ALBERT, 8 - SciBERT")
ch = input("Επιλογή: ")

# Επιλογή αριθμού εποχών εκπαίδευσης.
EPOCHS = int(input("Δώσε αριθμό εποχών εκπαίδευσης "))

# Δεκτές εποχές απο 3 έως 20.
while not EPOCHS >= 3 or not EPOCHS <=20:
    EPOCHS = int(input("Δώσε αριθμό εποχών εκπαίδευσης "))


if ch == "1":
    FOLDER_NAME = "BERT" # Κατάλογος όπου αποθηκεύεται το μοντέλο BERT.
    MODEL_NAME = "bert-base-uncased"
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3) #hidden_dropout_prob=0.3, attention_probs_dropout_prob=0.3
    print("Χρησιμοποιείται το BERT μοντέλο.")
elif ch == "2":
    FOLDER_NAME = "DeBERTa" # Κατάλογος όπου αποθηκεύεται το μοντέλο DeBERTa.
    MODEL_NAME = "microsoft/deberta-v3-base"
    tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)
    model = DebertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True)
    print("Χρησιμοποιείται το DeBERTa μοντέλο.")
elif ch == "3":
    FOLDER_NAME = "XLNet" # Κατάλογος όπου αποθηκεύεται το μοντέλο XLNet.
    MODEL_NAME = "xlnet-base-cased"
    tokenizer = XLNetTokenizer.from_pretrained(MODEL_NAME)
    model = XLNetForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    print("Χρησιμοποιείται το XLNet μοντέλο.")
elif ch == "4":
    FOLDER_NAME = "RoBERTa" # Κατάλογος όπου αποθηκεύεται το μοντέλο RoBERTa.
    MODEL_NAME = "roberta-base"
    tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
    model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3) #hidden_dropout_prob=0.3, attention_probs_dropout_prob=0.3
    print("Χρησιμοποιείται το RoBERTa μοντέλο.")
elif ch == "5":
    FOLDER_NAME = "DistilBERT" # Κατάλογος όπου αποθηκεύεται το μοντέλο DistilBERT.
    MODEL_NAME = "distilbert-base-uncased"
    tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
    model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    print("Χρησιμοποιείται το DistilBERT μοντέλο.")
elif ch == "6":
    FOLDER_NAME = "ELECTRA" # Κατάλογος όπου αποθηκεύεται το μοντέλο ELECTRA.
    MODEL_NAME = "google/electra-base-discriminator"
    tokenizer = ElectraTokenizer.from_pretrained(MODEL_NAME)
    model = ElectraForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    print("Χρησιμοποιείται το ELECTRA μοντέλο.")
elif ch == "7":
    FOLDER_NAME = "ALBERT" # Κατάλογος όπου αποθηκεύεται το μοντέλο ALBERT.
    MODEL_NAME = "albert-base-v2"
    tokenizer = AlbertTokenizer.from_pretrained(MODEL_NAME)
    model = AlbertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    print("Χρησιμοποιείται το ALBERT μοντέλο.")
elif ch == "8":
    FOLDER_NAME = "SciBERT" # Κατάλογος όπου αποθηκεύεται το μοντέλο SciBERT.
    MODEL_NAME = "allenai/scibert_scivocab_uncased"
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    print("Χρησιμοποιείται το SciBERT μοντέλο.")
else:
    raise ValueError("Μη έγκυρη επιλογή μοντέλου!")


# Προετοιμασία των δεδομένων.
train_encodings = tokenizer(train_data['sentence'].tolist(), truncation=True, padding=True, max_length=256)
test_encodings = tokenizer(test_data['sentence'].tolist(), truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(val_data['sentence'].tolist(), truncation=True, padding=True, max_length=256)

# Δημιουργία του dataset.
train_dataset = PolarityDataset(train_encodings, train_data['polarity'].tolist())
test_dataset = PolarityDataset(test_encodings, test_data['polarity'].tolist())
val_dataset = PolarityDataset(val_encodings, val_data['polarity'].tolist())


# Εμφάνιση των tokens του train_encodings.
input_ids = train_encodings['input_ids'][0]   # το πρώτο δείγμα του train μετά τον τεμαχισμό.
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print(tokens)


from transformers import EarlyStoppingCallback

# Ορίσματα εκπαίδευσης.
training_args = TrainingArguments(
    output_dir = './results',          # output directory
    run_name="Sentiment_Analysis_MODELS",
    num_train_epochs = EPOCHS,         # Αριθμός εποχών εκπαίδευσης.
    per_device_train_batch_size = 16,  # batch size για εκπαίδευση.
    per_device_eval_batch_size = 16,   # batch size για αξιολόγηση.
    #warmup_steps = 0,                # Ορίζει τον αριθμό των βημάτων εκπαίδευσης για τα οποία ο ρυθμός μάθησης (learning rate) αυξάνεται σταδιακά από το 0 μέχρι να φτάσει στην προκαθορισμένη τιμή του.
    weight_decay = 0.02,              # Τεχνική που εφαρμόζεται στον αλγόριθμο βελτιστοποίησης για να αποτρέψει την υπερπροσαρμογή (overfitting) του μοντέλου.
    logging_dir = './logs',            # directory for storing logs
    logging_steps = 50,
    warmup_ratio = 0.1,
    eval_strategy = "epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True,
    save_total_limit=2,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="adamw_torch_fused"
)

#my_optimizer = Adam(model.parameters(), lr=5e-5)  # Προσαρμογή του learning rate.
#my_optimizer = bnb.optim.Adam32bit(model.parameters(), lr=1e-5)
#my_optimizer = bnb.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01) #ηταν 3e-5


from transformers import TrainerCallback

"""class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        self.train_losses = []
        self.eval_losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if "loss" in logs:
                self.train_losses.append(logs["loss"])
            if "eval_loss" in logs:
                self.eval_losses.append(logs["eval_loss"])
"""

class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        # θα κρατήσουμε όλα τα logs με epoch, loss, eval_loss
        self.records = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        epoch = logs.get("epoch", None)
        loss = logs.get("loss", None)
        eval_loss = logs.get("eval_loss", None)

        # κρατάμε μόνο logs που έχουν epoch ΚΑΙ τουλάχιστον ένα από loss / eval_loss
        if epoch is not None and (loss is not None or eval_loss is not None):
            self.records.append({
                "epoch": epoch,
                "loss": loss,
                "eval_loss": eval_loss
            })



# Δημιουργία callback
loss_logger = LossLoggerCallback()

# Εκπαίδευση του μοντέλου με custom callback για logging του Loss.
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    #optimizers=(my_optimizer, None),  # None σημαίνει ότι χρησιμοποιείται το default scheduler.
    callbacks=[loss_logger, EarlyStoppingCallback(early_stopping_patience=3)]  # Προσθήκη του callback..
)

# Ορισμός του χρόνου έναρξης
start_time = time.time()

# Εκπαίδευση του μοντέλου
print("Ξεκινά η εκπαίδευση του μοντέλου...\n")
trainer.train()

# Ορισμός του χρόνου λήξης
end_time = time.time()

# Υπολογισμός του συνολικού χρόνου εκπαίδευσης
total_time = end_time - start_time

# Μετατροπή σε λεπτά & δευτερόλεπτα.
# Χρησιμοποιούμε // 60 για να βρούμε τα λεπτά και % 60 για τα υπόλοιπα δευτερόλεπτα.
# Το % είναι ο τελεστής υπολοίπου MOD.
# Το // κάνει ακέραια διαίρεση, επιστρέφοντας μόνο το ακέραιο μέρος.
minutes = int(total_time // 60)
seconds = int(total_time % 60)
print(f"\nΣυνολικός Χρόνος Εκπαίδευσης: {minutes} λεπτά και {seconds} δευτερόλεπτα ({training_args.num_train_epochs} εποχές)")


### Οπτικοποίηση του Training & Validation Loss ###
#plt.figure(figsize=(7, 5))
#plt.plot(loss_logger.epochs, loss_logger.train_losses, label="Training", marker='o')
#plt.plot(loss_logger.epochs, loss_logger.val_losses, label="Validation", marker='X')
#plt.xlabel("Epochs")
#plt.ylabel("Loss")
#plt.title("Training & Validation Loss")
#plt.legend()
#plt.grid(True)
#plt.show()


# Τα raw logs από το callback
records = loss_logger.records

df_logs = pd.DataFrame(records)

# Διαχωρίζουμε train και val
train_df = df_logs[df_logs["loss"].notna()][["epoch", "loss"]]
train_df = train_df.rename(columns={"loss": "train_loss"})

eval_df = df_logs[df_logs["eval_loss"].notna()][["epoch", "eval_loss"]]
eval_df = eval_df.rename(columns={"eval_loss": "val_loss"})

# Κάνουμε merge ανά epoch
df_loss = pd.merge(train_df, eval_df, on="epoch", how="left")

# Τακτοποιούμε / στρογγυλοποιούμε
df_loss = df_loss.sort_values("epoch").reset_index(drop=True)
df_loss["train_loss"] = df_loss["train_loss"].astype(float)
df_loss["val_loss"] = df_loss["val_loss"].astype(float)

print("To CSV είναι: ")
print(df_loss)

# Αποθήκευση σε CSV για αυτό το μοντέλο
csv_path = f"data/processed/logs/loss_{FOLDER_NAME}.csv"
df_loss.to_csv(csv_path, index=False)
print("Saved loss CSV to:", csv_path)

In [ ]:
#from mpl_toolkits.axes_grid1.inset_locator import inset_axes
"""from matplotlib.ticker import FormatStrFormatter

# Δεδομένα
epochs = list(range(1, len(loss_logger.eval_losses) + 1))

df = pd.DataFrame({
    "Epoch": epochs + epochs,
    "Loss": loss_logger.train_losses + loss_logger.eval_losses,
    "Type": ["Train"] * len(epochs) + ["Validation"] * len(epochs)
})

# Custom markers
markers = {"Train": "D", "Validation": "X"}

# Custom colors
palette = {"Train": "tab:blue", "Validation": "tab:orange"}

# Plot κύριο διάγραμμα

fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
#plt.figure(figsize=(6, 4))
plt.rcParams.update({'font.size': 11})

sns.set_style("whitegrid")

sns.lineplot(
    data=df, x="Epoch", y="Loss",
    hue="Type", style="Type",
    markers=markers, palette=palette,
    linewidth=1, markersize=7,
    markeredgecolor="white", markeredgewidth=1.2,
    dashes=False,
    ax=ax
)

# Format στον άξονα y -> 2 δεκαδικά
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
ax.set_yticks(np.linspace(0, 1, 6))


# --- δίνουμε λίγο "αέρα" στον y-άξονα του κύριου plot ---
ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin - 0.02, ymax + 0.02)


ax.legend(
    title=None
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(epochs)  # Ticks σε όλα τα epochs
plt.grid(True, linestyle="--", alpha=0.7)


# --- Αποθήκευση PDF ---
plt.savefig(
    f"outputs/figures/fig/{FOLDER_NAME}_loss_plot_{EPOCHS}_epochs.pdf",
    format='pdf',
    bbox_inches='tight',
    dpi=300
)

plt.show()"""


In [ ]:
# Αποθήκευση του μοντέλου και του tokenizer (για πολικότητα).
model.save_pretrained(f'external_materials/model_weights/{FOLDER_NAME}/saved_model')
tokenizer.save_pretrained(f'external_materials/model_weights/{FOLDER_NAME}/saved_tokenizer')

print(f"\nΤο fine-tuned μοντέλο ({MODEL_NAME}) αποθηκεύτηκε επιτυχώς!")

# Προβλέψεις με το test set
predictions = trainer.predict(test_dataset)
predicted_classes = np.argmax(predictions.predictions, axis=-1)

# Classification Report
report = classification_report(test_data['polarity'], predicted_classes, target_names=['Negative', 'Neutral', 'Positive'])
print(report)

with open(f"outputs/results/classification_reports/{FOLDER_NAME}_classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

# Heatmap
# Ενοποιημένο μέγεθος γραμματοσειράς
plt.rcParams.update({'font.size': 11})
plt.figure(figsize=(6, 4))

cm = confusion_matrix(test_data['polarity'], predicted_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Neutral', 'Positive'], yticklabels=['Negative', 'Neutral', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()

#plt.title(f'Confusion Matrix - {FOLDER_NAME}')

# Αποθήκευση ως PDF υψηλής ανάλυσης.
plt.savefig(
    f'outputs/figures/fig/{FOLDER_NAME}_heatmap_SA_citation.pdf',
    format='pdf',
    bbox_inches='tight',
    dpi=300
)

plt.show()

# cmap = magma, cividis, Greys, Purples, Blues, Greens, Oranges, Reds

In [ ]:
# Αποθήκευση των δεδομένων train_data['sentence'], test_data['sentence'] και val_data['sentence']
# σε αρχεία προκειμένου να επιβεβαιώσουμε ότι σε κάθε εκτέλεση του συγκεκριμένου σημειωματαρίου
# τα δεδομένα μας θα έιναι πάντα τα ίδια. Δηλαδή ίδιο train set, ίδιο test set και ίδιο validation set.
# Έτσι είναι πιο ακριβής η σύγκριση των μοντέλων.

# Αποθήκευση των δεδομένων σε αρχεία .txt
train_data['sentence'].to_csv('train_sentences.txt', index=False, header=False)
test_data['sentence'].to_csv('test_sentences.txt', index=False, header=False)
val_data['sentence'].to_csv('val_sentences.txt', index=False, header=False)

print("Τα αρχεία αποθηκεύτηκαν επιτυχώς!")


In [ ]:
# Κατέβασμα αρχείων.
from google.colab import files

files.download('train_sentences.txt')
files.download('test_sentences.txt')
files.download('val_sentences.txt')
